# Sese 10 - Laporan Otomatis dengan Python

Kita akan generate laporan otomatis ke dalam Excel berdasarkan hasil analisis
Skill ini akan berguna di dunia kerja
Lapran yang biasa dibuat berjam-jam akan dibuat otomasi dengan Python dalam hitungan detik

## 1. Persiapan dan Install Library

In [5]:
import subprocess
subprocess.run(["pip", "install", "openpyxl"], capture_output=True)

CompletedProcess(args=['pip', 'install', 'openpyxl'], returncode=0, stdout=b'Requirement already satisfied: openpyxl in /opt/anaconda3/lib/python3.12/site-packages (3.1.5)\nRequirement already satisfied: et-xmlfile in /opt/anaconda3/lib/python3.12/site-packages (from openpyxl) (1.1.0)\n', stderr=b'')

In [7]:
import pandas as pd
import matplotlib.pyplot as plt
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils.dataframe import dataframe_to_rows

In [9]:
df = pd.read_csv("samplesuperstore.csv", encoding="latin1")
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Year"] = df["Order Date"].dt.year

print("Data Siap!")
print("Shape: ", df.shape)

Data Siap!
Shape:  (10194, 22)


## 2. Siapkan Data Ringkasan

In [11]:
# Ringkasan per kategori
per_kategori = df.groupby("Category").agg(
    Total_Sales = ("Sales", "sum"),
    Total_Profit = ("Profit", "sum"),
    Jumlah_Order = ("Order ID", "count")
).reset_index()
per_kategori["Margin_%"] = (per_kategori["Total_Profit"]/per_kategori["Total_Sales"] * 100).round(1)

#Ringkasan per region
per_region = df.groupby("Region").agg(
    Total_Sales = ("Sales", "sum"),
    Total_Profit = ("Profit", "sum")
).reset_index()

print(per_kategori)
print()
print(per_region)

          Category  Total_Sales  Total_Profit  Jumlah_Order  Margin_%
0        Furniture  754747.7613    19729.9956          2201       2.6
1  Office Supplies  731893.3140   126023.4434          6128      17.2
2       Technology  839893.2790   146543.3756          1865      17.4

    Region  Total_Sales  Total_Profit
0  Central  503170.6728    39865.3070
1     East  691828.1680    94883.2603
2    South  391721.9050    46749.4303
3     West  739813.6085   110798.8170


## 3. Generate Laporan Excel

In [15]:
wb = Workbook()

# ===== SHEET 1: RINGKASAN KATEGORI =====
ws1 = wb.active
ws1.title = "Per Kategori"

# Header
ws1["A1"] = "LAPORAN ANALISIS PENJUALAN - SUPERSTORE"
ws1["A1"].font = Font(bold=True, size=15)
ws1["A3"] = "Ringkasan per Kategori"
ws1["A3"].font = Font(bold=True, size=12)

# Tulis Data
for r in dataframe_to_rows(per_kategori, index=False, header=True):
    ws1.append(r)

# Warnai header tabel
header_fill = PatternFill(fill_type="solid", fgColor="4472C4")
for cell in ws1[4]:
    cell.fill = header_fill
    cell.font = Font(bold=True, color="FFFFFF")
    cell.alignment = Alignment(horizontal = "center")

# Auto width kolom
for col in ws1.columns:
    max_len = max(len(str(cell.value or "")) for cell in col)
    ws1.column_dimensions[col[0].column_letter].width = max_len + 4

# ===== SHEET 2: RINGKASAN REGION =====
ws2 = wb.create_sheet("Per Region")
ws2["A1"] = "Ringkasan per Region"
ws2["A1"].font = Font(bold=True, size=15)

for r in dataframe_to_rows(per_region, index=False, header=True):
    ws2.append(r)

for cell in ws2[2]:
    cell.fill = PatternFill(fill_type="solid", fgColor="70AD47")
    cell.font = Font(bold=True, color="FFFFFF")
    cell.alignment = Alignment(horizontal="center")

for col in ws2.columns:
    max_len = max(len(str(cell.value or "")) for cell in col)
    ws2.column_dimensions[col[0].column_letter].width = max_len + 4

# Simpan File
wb.save("laporan-superstore.xlsx")
print("laporan Excel berhasil dibuat: laporan_superstore.xlsx")

laporan Excel berhasil dibuat: laporan_superstore.xlsx
